# Repeat Purchase Behavior Analysis
### Studying purchase frequency, repeat rates, interpurchase timing, and retention behavior

**Project goal:** identify which customer groups are more likely to repurchase, how quickly they come back, and what first-order behaviors are associated with stronger repeat demand.

## Project Overview

This notebook uses the Online Retail transaction log to move from line-item sales to customer-level repeat behavior. The analysis focuses on four business questions:

1. How often do customers come back after the first order?
2. How large is the repeat-customer base relative to one-time buyers?
3. How much time usually passes between purchases?
4. Which customer groups show stronger retention, and what early signals explain that pattern?

The goal is not just to count repeat customers, but to convert transaction history into actionable retention insights.

## Learning Objectives

1. Convert raw line-item invoices into clean customer-order histories.
2. Measure repeat rate, three-plus-order rate, and repurchase timing directly from transactions.
3. Build monthly retention cohorts and interpret cohort decay.
4. Compare repeat behavior across geography and first-order basket characteristics.
5. Distinguish between descriptive frequency metrics and customer-level retention behavior.
6. Translate repeat-purchase patterns into concrete commercial recommendations.

## Business / Research Problem

Customer growth is expensive when most first-time buyers never return. The core problem is to quantify repeat behavior accurately and then isolate which customer groups are most likely to repurchase so retention efforts can focus on the right customers at the right moment.

## Why This Analysis Matters

Repeat purchasing compounds revenue, lowers acquisition payback time, and usually improves gross margin because future orders arrive without reacquiring the same customer. Understanding interpurchase timing is equally important: if a second order usually happens inside a short window, delayed follow-up can miss the highest-propensity moment.

## Dataset Overview

The notebook uses the repo-local Online Retail workbook already present in this workspace.

- **Source file:** `Online Retail.xlsx`
- **Grain:** one row per product line inside an invoice
- **Core columns:** `InvoiceNo`, `InvoiceDate`, `CustomerID`, `Quantity`, `UnitPrice`, `Country`
- **Customer grouping options available in this local copy:** geography plus first-order basket value, breadth, and unit volume derived from transactions
- **Important note:** the local file also includes a helper column named `lower`; it is not part of the business logic and is removed during cleaning

## Dataset Source and License Notes

This workbook is loaded from the repository rather than downloaded at runtime. The broader Online Retail dataset is commonly distributed through the UCI / Kaggle ecosystem. Before external reuse, verify the license terms attached to the specific copy you rely on.

## Environment Setup

In [1]:
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    "openpyxl": "openpyxl",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "scipy": "scipy",
}

for module_name, package_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

print("Environment ready.")

Environment ready.


## Imports

In [2]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.stats import chi2_contingency, mannwhitneyu, spearmanr

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

print("Imports ready.")

Imports ready.


## Configuration / Constants

In [3]:
DATA_CANDIDATES = [
    Path("Data Analysis/CLV Online Retail/Online Retail.xlsx"),
    Path("../CLV Online Retail/Online Retail.xlsx"),
    Path("Online Retail.xlsx"),
]
TOP_N_COUNTRIES = 8
MIN_COUNTRY_CUSTOMERS = 30
RETENTION_PERIODS = 6

def resolve_data_path(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not locate Online Retail.xlsx in the expected repo paths.")

def quantile_segment(series, labels):
    series = series.astype(float)
    ranked = series.rank(method="first")
    unique_values = ranked.nunique(dropna=True)
    bins = min(len(labels), int(unique_values))

    if bins <= 1:
        category = labels[0] if labels else "All"
        return pd.Series(
            pd.Categorical([category] * len(series), categories=[category], ordered=True),
            index=series.index,
        )

    active_labels = labels[:bins]
    segmented = pd.qcut(ranked, q=bins, labels=active_labels)
    return pd.Series(pd.Categorical(segmented, categories=active_labels, ordered=True), index=series.index)

DATA_PATH = resolve_data_path(DATA_CANDIDATES)
print(f"Resolved data path: {DATA_PATH}")

Resolved data path: ..\CLV Online Retail\Online Retail.xlsx


## Dataset Loading

In [4]:
raw = pd.read_excel(DATA_PATH)
print(f"Raw shape: {raw.shape}")
print(f"Columns: {list(raw.columns)}")
display(raw.head())

Raw shape: (541909, 9)
Columns: ['InvoiceNo', 'StockCode', 'lower', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


,InvoiceNo,StockCode,lower,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,white hanging heart t-light holder,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,white metal lantern,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,cream cupid hearts coat hanger,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom
3,536365,84029G,knitted union flag hot water bottle,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
4,536365,84029E,red woolly hottie white heart.,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom


## Data Validation Checks

Before measuring retention, verify that the file really contains the ingredients required for repeat-purchase analysis: customer IDs, invoice IDs, timestamps, positive quantities, and usable price information.

In [5]:
validation = raw.copy()
validation["InvoiceNo"] = validation["InvoiceNo"].astype(str)
validation["InvoiceDate"] = pd.to_datetime(validation["InvoiceDate"], errors="coerce")

validation_report = pd.Series(
    {
        "Rows": len(validation),
        "Unique invoices": validation["InvoiceNo"].nunique(dropna=True),
        "Unique customers": validation["CustomerID"].nunique(dropna=True),
        "Missing CustomerID": validation["CustomerID"].isna().sum(),
        "Missing InvoiceDate": validation["InvoiceDate"].isna().sum(),
        "Cancellation invoices": validation["InvoiceNo"].str.startswith("C", na=False).sum(),
        "Non-positive quantity rows": (validation["Quantity"] <= 0).sum(),
        "Non-positive price rows": (validation["UnitPrice"] <= 0).sum(),
        "Date range start": validation["InvoiceDate"].min(),
        "Date range end": validation["InvoiceDate"].max(),
    },
    name="Validation",
)
display(validation_report.to_frame())

,Validation
Rows,541909
Unique invoices,25900
Unique customers,4372
Missing CustomerID,135080
Missing InvoiceDate,0
Cancellation invoices,9288
Non-positive quantity rows,10624
Non-positive price rows,2517
Date range start,2010-12-01 08:26:00
Date range end,2011-12-09 12:50:00


## Data Cleaning And Customer / Order Preparation

Repeat analysis should operate at the **order** level, not the line-item level. We remove cancellations, records without customer identity, and invalid quantity or price rows, then aggregate all product lines inside each invoice into a single customer order.

In [6]:
df = raw.copy()
df = df.drop(columns=[column for column in ["lower"] if column in df.columns])
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df = df.dropna(subset=["CustomerID", "InvoiceDate"])
df = df[~df["InvoiceNo"].str.startswith("C", na=False)].copy()
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()
df["CustomerID"] = df["CustomerID"].astype(int).astype(str)
df["Sales"] = df["Quantity"] * df["UnitPrice"]

orders = (
    df.groupby(["CustomerID", "InvoiceNo"], as_index=False)
    .agg(
        InvoiceDate=("InvoiceDate", "min"),
        OrderSales=("Sales", "sum"),
        Units=("Quantity", "sum"),
        SKUCount=("StockCode", "nunique"),
        LineCount=("StockCode", "count"),
        Country=("Country", lambda series: series.mode().iat[0] if not series.mode().empty else series.iloc[0]),
    )
    .sort_values(["CustomerID", "InvoiceDate", "InvoiceNo"])
    .reset_index(drop=True)
)

orders["OrderRank"] = orders.groupby("CustomerID").cumcount() + 1
orders["GapDays"] = orders.groupby("CustomerID")["InvoiceDate"].diff().dt.days
orders["PurchaseMonth"] = orders["InvoiceDate"].dt.to_period("M").astype(str)
first_purchase = orders.groupby("CustomerID")["InvoiceDate"].transform("min")
orders["CohortMonth"] = first_purchase.dt.to_period("M").astype(str)
orders["CohortIndex"] = (
    (orders["InvoiceDate"].dt.year - first_purchase.dt.year) * 12
    + (orders["InvoiceDate"].dt.month - first_purchase.dt.month)
)

first_order_features = (
    orders.loc[orders["OrderRank"] == 1, ["CustomerID", "OrderSales", "Units", "SKUCount"]]
    .rename(
        columns={
            "OrderSales": "FirstOrderValue",
            "Units": "FirstOrderUnits",
            "SKUCount": "FirstOrderSkus",
        }
    )
    .copy()
)

customer_summary = (
    orders.groupby("CustomerID", as_index=False)
    .agg(
        Country=("Country", lambda series: series.mode().iat[0] if not series.mode().empty else series.iloc[0]),
        OrderCount=("InvoiceNo", "nunique"),
        Revenue=("OrderSales", "sum"),
        AvgOrderValue=("OrderSales", "mean"),
        MedianOrderValue=("OrderSales", "median"),
        AvgUnitsPerOrder=("Units", "mean"),
        AvgSkusPerOrder=("SKUCount", "mean"),
        FirstPurchaseDate=("InvoiceDate", "min"),
        LastPurchaseDate=("InvoiceDate", "max"),
        MeanGapDays=("GapDays", "mean"),
        MedianGapDays=("GapDays", "median"),
        MinGapDays=("GapDays", "min"),
        MaxGapDays=("GapDays", "max"),
        ActiveMonths=("PurchaseMonth", "nunique"),
    )
    .merge(first_order_features, on="CustomerID", how="left")
)

customer_summary["CustomerLifetimeDays"] = (
    customer_summary["LastPurchaseDate"] - customer_summary["FirstPurchaseDate"]
).dt.days
customer_summary["RepeatCustomer"] = customer_summary["OrderCount"] >= 2
customer_summary["ThreePlusOrders"] = customer_summary["OrderCount"] >= 3
customer_summary["RepeatWithin30Days"] = customer_summary["MinGapDays"].le(30).fillna(False)
customer_summary["RepeatWithin60Days"] = customer_summary["MinGapDays"].le(60).fillna(False)
customer_summary["RepeatWithin90Days"] = customer_summary["MinGapDays"].le(90).fillna(False)
customer_summary["MonthlyOrderRate"] = customer_summary["OrderCount"] / customer_summary["ActiveMonths"].clip(lower=1)
customer_summary["RevenuePerActiveMonth"] = customer_summary["Revenue"] / customer_summary["ActiveMonths"].clip(lower=1)
customer_summary["Country"] = customer_summary["Country"].fillna("Unknown")

top_countries = customer_summary["Country"].value_counts().head(TOP_N_COUNTRIES).index
customer_summary["CountryGroup"] = np.where(
    customer_summary["Country"].isin(top_countries), customer_summary["Country"], "Other"
)
customer_summary["FirstOrderValueSegment"] = quantile_segment(
    customer_summary["FirstOrderValue"], ["Low", "Mid", "High"]
)
customer_summary["FirstOrderBreadthSegment"] = quantile_segment(
    customer_summary["FirstOrderSkus"], ["Narrow", "Balanced", "Wide"]
)
customer_summary["AOVSegment"] = quantile_segment(
    customer_summary["AvgOrderValue"], ["Low AOV", "Mid AOV", "High AOV"]
)
customer_summary["TenureSegment"] = pd.cut(
    customer_summary["CustomerLifetimeDays"],
    bins=[-1, 0, 30, 90, 180, 10000],
    labels=["Single day", "1-30d", "31-90d", "91-180d", "180d+"],
    include_lowest=True,
)

print(f"Clean order table shape: {orders.shape}")
print(f"Customer summary shape: {customer_summary.shape}")
display(customer_summary.head())

Clean order table shape: (18532, 13)
Customer summary shape: (4338, 31)


,CustomerID,Country,OrderCount,Revenue,AvgOrderValue,MedianOrderValue,AvgUnitsPerOrder,AvgSkusPerOrder,FirstPurchaseDate,LastPurchaseDate,MeanGapDays,MedianGapDays,MinGapDays,MaxGapDays,ActiveMonths,FirstOrderValue,FirstOrderUnits,FirstOrderSkus,CustomerLifetimeDays,RepeatCustomer,ThreePlusOrders,RepeatWithin30Days,RepeatWithin60Days,RepeatWithin90Days,MonthlyOrderRate,RevenuePerActiveMonth,CountryGroup,FirstOrderValueSegment,FirstOrderBreadthSegment,AOVSegment,TenureSegment
0,12346,United Kingdom,1,"77,183.60","77,183.60","77,183.60","74,215.00",1.00,2011-01-18 10:01:00,2011-01-18 10:01:00,NaN,NaN,NaN,NaN,1,"77,183.60",74215,1,0,False,False,False,False,False,1.00,"77,183.60",United Kingdom,High,Narrow,High AOV,Single day
1,12347,Iceland,7,"4,310.00",615.71,584.91,351.14,26.00,2010-12-07 14:57:00,2011-12-07 15:52:00,60.33,58.00,37.00,90.00,7,711.79,319,31,365,True,True,False,True,True,1.00,615.71,Other,High,Wide,High AOV,180d+
2,12348,Finland,4,"1,797.24",449.31,338.50,585.25,6.75,2010-12-16 19:09:00,2011-09-25 13:13:00,94.00,70.00,39.00,173.00,4,892.80,1254,13,282,True,True,False,True,True,1.00,449.31,Other,High,Balanced,High AOV,180d+
3,12349,Italy,1,"1,757.55","1,757.55","1,757.55",631.00,73.00,2011-11-21 09:51:00,2011-11-21 09:51:00,NaN,NaN,NaN,NaN,1,"1,757.55",631,73,0,False,False,False,False,False,1.00,"1,757.55",Italy,High,Wide,High AOV,Single day
4,12350,Norway,1,334.40,334.40,334.40,197.00,17.00,2011-02-02 16:01:00,2011-02-02 16:01:00,NaN,NaN,NaN,NaN,1,334.40,197,17,0,False,False,False,False,False,1.00,334.40,Other,Mid,Balanced,Mid AOV,Single day


## Exploratory Data Analysis

Start with the headline repeat metrics so later group comparisons stay anchored to the overall customer base and not just to selected segments.

In [7]:
gap_days = orders["GapDays"].dropna()

scorecard = pd.Series(
    {
        "Customers": len(customer_summary),
        "Orders": len(orders),
        "Revenue": customer_summary["Revenue"].sum(),
        "Repeat rate (%)": customer_summary["RepeatCustomer"].mean() * 100,
        "3+ orders rate (%)": customer_summary["ThreePlusOrders"].mean() * 100,
        "Median orders per customer": customer_summary["OrderCount"].median(),
        "Median interpurchase gap (days)": gap_days.median(),
        "90th percentile gap (days)": gap_days.quantile(0.90),
        "Repeat within 30 days (%)": customer_summary["RepeatWithin30Days"].mean() * 100,
        "Repeat within 90 days (%)": customer_summary["RepeatWithin90Days"].mean() * 100,
    },
    name="Scorecard",
)

display(scorecard.to_frame().round(2))

,Scorecard
Customers,"4,338.00"
Orders,"18,532.00"
Revenue,"8,911,407.90"
Repeat rate (%),65.58
3+ orders rate (%),46.33
Median orders per customer,2.00
Median interpurchase gap (days),21.00
90th percentile gap (days),102.70
Repeat within 30 days (%),39.79
Repeat within 90 days (%),54.31


## Univariate Analysis

Univariate views show how concentrated repeat behavior is. A healthy repeat business still may be highly unequal if a small minority of customers accounts for most orders.

In [8]:
order_count_distribution = (
    customer_summary["OrderCount"].value_counts().sort_index().rename_axis("OrderCount").reset_index(name="Customers")
)
gap_distribution = customer_summary.loc[customer_summary["RepeatCustomer"], "MedianGapDays"].dropna()

print("Order-count distribution (first 10 buckets):")
display(order_count_distribution.head(10))

summary_stats = customer_summary[[
    "OrderCount",
    "Revenue",
    "AvgOrderValue",
    "FirstOrderValue",
    "FirstOrderSkus",
    "CustomerLifetimeDays",
]].describe().T
display(summary_stats)

print(f"Median customer-level gap among repeat customers: {gap_distribution.median():.1f} days")

Order-count distribution (first 10 buckets):


,OrderCount,Customers
0,1,1493
1,2,835
2,3,508
3,4,388
4,5,242
5,6,172
6,7,143
7,8,98
8,9,68
9,10,54


,count,mean,std,min,25%,50%,75%,max
OrderCount,"4,338.00",4.27,7.70,1.00,1.00,2.00,5.00,209.00
Revenue,"4,338.00","2,054.27","8,989.23",3.75,307.41,674.49,"1,661.74","280,206.02"
AvgOrderValue,"4,338.00",419.17,"1,796.54",3.45,178.62,293.90,430.11,"84,236.25"
FirstOrderValue,"4,338.00",425.66,"1,290.13",0.85,167.37,301.12,449.42,"77,183.60"
FirstOrderSkus,"4,338.00",23.30,22.30,1.00,9.00,17.00,30.00,262.00
CustomerLifetimeDays,"4,338.00",130.45,132.04,0.00,0.00,92.50,251.75,373.00


Median customer-level gap among repeat customers: 46.0 days


## Bivariate / Multivariate Analysis

This section compares repeat behavior across customer groups. The focus is on geography, acquisition cohort, and first-order basket structure because those dimensions can guide both marketing and merchandising actions.

In [9]:
country_repeat = (
    customer_summary.groupby("CountryGroup", as_index=False)
    .agg(
        Customers=("CustomerID", "count"),
        RepeatRatePct=("RepeatCustomer", lambda values: values.mean() * 100),
        ThreePlusRatePct=("ThreePlusOrders", lambda values: values.mean() * 100),
        MedianGapDays=("MedianGapDays", "median"),
        AvgRevenue=("Revenue", "mean"),
    )
    .sort_values(["Customers", "RepeatRatePct"], ascending=[False, False])
    .reset_index(drop=True)
)

customer_summary["AcquisitionQuarter"] = customer_summary["FirstPurchaseDate"].dt.to_period("Q").astype(str)
cohort_repeat = (
    customer_summary.groupby("AcquisitionQuarter", as_index=False)
    .agg(
        Customers=("CustomerID", "count"),
        RepeatRatePct=("RepeatCustomer", lambda values: values.mean() * 100),
        MedianOrders=("OrderCount", "median"),
        MedianGapDays=("MedianGapDays", "median"),
    )
    .sort_values("AcquisitionQuarter")
)

display(country_repeat.head(12).round(2))
display(cohort_repeat.round(2))

,CountryGroup,Customers,RepeatRatePct,ThreePlusRatePct,MedianGapDays,AvgRevenue
0,United Kingdom,3920,65.56,46.53,46.00,"1,864.39"
1,Other,131,59.54,37.40,49.50,"7,277.80"
2,Germany,94,72.34,53.19,41.50,"2,434.76"
3,France,87,67.82,52.87,42.00,"2,402.58"
4,Spain,29,65.52,37.93,82.00,"2,166.89"
5,Belgium,24,75.00,54.17,46.50,"1,731.30"
6,Switzerland,20,75.00,35.00,40.00,"2,820.96"
7,Portugal,19,63.16,36.84,40.50,"1,759.99"
8,Italy,14,42.86,21.43,25.00,"1,248.80"


,AcquisitionQuarter,Customers,RepeatRatePct,MedianOrders,MedianGapDays
0,2010Q4,885,87.46,6.00,41.00
1,2011Q1,1249,76.78,3.00,60.00
2,2011Q2,826,66.95,2.00,57.00
3,2011Q3,656,53.20,2.00,36.00
4,2011Q4,722,29.09,1.00,15.00


## Feature-Specific Insights

To explain *why* some customers repurchase more often, we compare repeat behavior against first-order basket value and product breadth. These are usable commercial signals because they are visible immediately after the first purchase.

In [10]:
value_segment_table = (
    customer_summary.groupby("FirstOrderValueSegment", observed=True, as_index=False)
    .agg(
        Customers=("CustomerID", "count"),
        RepeatRatePct=("RepeatCustomer", lambda values: values.mean() * 100),
        ThreePlusRatePct=("ThreePlusOrders", lambda values: values.mean() * 100),
        MedianGapDays=("MedianGapDays", "median"),
        AvgRevenue=("Revenue", "mean"),
    )
)

breadth_segment_table = (
    customer_summary.groupby("FirstOrderBreadthSegment", observed=True, as_index=False)
    .agg(
        Customers=("CustomerID", "count"),
        RepeatRatePct=("RepeatCustomer", lambda values: values.mean() * 100),
        ThreePlusRatePct=("ThreePlusOrders", lambda values: values.mean() * 100),
        MedianGapDays=("MedianGapDays", "median"),
        AvgRevenue=("Revenue", "mean"),
    )
)

driver_table = pd.DataFrame(
    {
        "Metric": [
            "First order value",
            "First order SKUs",
            "Average order value",
            "Monthly order rate",
            "Revenue per active month",
        ],
        "One-time median": [
            customer_summary.loc[~customer_summary["RepeatCustomer"], "FirstOrderValue"].median(),
            customer_summary.loc[~customer_summary["RepeatCustomer"], "FirstOrderSkus"].median(),
            customer_summary.loc[~customer_summary["RepeatCustomer"], "AvgOrderValue"].median(),
            customer_summary.loc[~customer_summary["RepeatCustomer"], "MonthlyOrderRate"].median(),
            customer_summary.loc[~customer_summary["RepeatCustomer"], "RevenuePerActiveMonth"].median(),
        ],
        "Repeat median": [
            customer_summary.loc[customer_summary["RepeatCustomer"], "FirstOrderValue"].median(),
            customer_summary.loc[customer_summary["RepeatCustomer"], "FirstOrderSkus"].median(),
            customer_summary.loc[customer_summary["RepeatCustomer"], "AvgOrderValue"].median(),
            customer_summary.loc[customer_summary["RepeatCustomer"], "MonthlyOrderRate"].median(),
            customer_summary.loc[customer_summary["RepeatCustomer"], "RevenuePerActiveMonth"].median(),
        ],
    }
)
driver_table["Relative lift (%)"] = (
    (driver_table["Repeat median"] / driver_table["One-time median"]) - 1
) * 100

segment_matrix = (
    customer_summary.pivot_table(
        index="FirstOrderBreadthSegment",
        columns="FirstOrderValueSegment",
        values="RepeatCustomer",
        aggfunc="mean",
        observed=True,
    )
    * 100
)

display(value_segment_table.round(2))
display(breadth_segment_table.round(2))
display(driver_table.round(2))
display(segment_matrix.round(1))

,FirstOrderValueSegment,Customers,RepeatRatePct,ThreePlusRatePct,MedianGapDays,AvgRevenue
0,Low,1446,59.27,39.28,51.00,"1,249.63"
1,Mid,1446,67.22,46.33,48.00,"1,335.32"
2,High,1446,70.26,53.39,42.00,"3,577.85"


,FirstOrderBreadthSegment,Customers,RepeatRatePct,ThreePlusRatePct,MedianGapDays,AvgRevenue
0,Narrow,1446,61.07,42.05,48.00,"2,287.46"
1,Balanced,1446,67.36,47.72,48.00,"1,731.08"
2,Wide,1446,68.33,49.24,43.50,"2,144.26"


,Metric,One-time median,Repeat median,Relative lift (%)
0,First order value,257.68,307.68,19.40
1,First order SKUs,16.00,18.00,12.50
2,Average order value,257.68,303.72,17.87
3,Monthly order rate,1.00,1.00,0.00
4,Revenue per active month,257.68,360.92,40.07


FirstOrderValueSegment,Low,Mid,High
FirstOrderBreadthSegment,,,
Narrow,57.60,66.30,68.50
Balanced,65.10,67.70,68.50
Wide,58.10,67.10,71.40


## Statistical Checks

The goal here is to test whether the observed differences are just visual noise or whether repeat customers truly look different from one-time buyers.

In [11]:
repeat_first_order = customer_summary.loc[customer_summary["RepeatCustomer"], "FirstOrderValue"]
one_time_first_order = customer_summary.loc[~customer_summary["RepeatCustomer"], "FirstOrderValue"]
repeat_first_skus = customer_summary.loc[customer_summary["RepeatCustomer"], "FirstOrderSkus"]
one_time_first_skus = customer_summary.loc[~customer_summary["RepeatCustomer"], "FirstOrderSkus"]

value_test = mannwhitneyu(repeat_first_order, one_time_first_order, alternative="two-sided")
sku_test = mannwhitneyu(repeat_first_skus, one_time_first_skus, alternative="two-sided")
country_contingency = pd.crosstab(customer_summary["CountryGroup"], customer_summary["RepeatCustomer"])
chi2_stat, chi2_pvalue, _, _ = chi2_contingency(country_contingency)
spearman_stat, spearman_pvalue = spearmanr(customer_summary["FirstOrderSkus"], customer_summary["OrderCount"])

stats_table = pd.DataFrame(
    {
        "Test": [
            "Mann-Whitney: first-order value by repeat status",
            "Mann-Whitney: first-order SKU breadth by repeat status",
            "Chi-square: country group vs repeat status",
            "Spearman: first-order SKU breadth vs order count",
        ],
        "Statistic": [value_test.statistic, sku_test.statistic, chi2_stat, spearman_stat],
        "p_value": [value_test.pvalue, sku_test.pvalue, chi2_pvalue, spearman_pvalue],
    }
)
display(stats_table)

,Test,Statistic,p_value
0,Mann-Whitney: first-order value by repeat status,"2,378,710.00",0.00
1,Mann-Whitney: first-order SKU breadth by repea...,"2,291,467.50",0.00
2,Chi-square: country group vs repeat status,9.19,0.33
3,Spearman: first-order SKU breadth vs order count,0.06,0.00


## Visual Analysis

Visuals make the commercial story easier to absorb: where repeat rates are highest, how quickly second orders usually arrive, and how retention decays by cohort after acquisition.

In [12]:
retention_counts = orders.groupby(["CohortMonth", "CohortIndex"])["CustomerID"].nunique().unstack(fill_value=0)
cohort_sizes = retention_counts.iloc[:, 0]
retention_rates = retention_counts.div(cohort_sizes, axis=0).iloc[:, : RETENTION_PERIODS + 1]

country_focus = country_repeat[country_repeat["Customers"] >= MIN_COUNTRY_CUSTOMERS].copy()

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

sns.barplot(
    data=country_focus.sort_values("RepeatRatePct", ascending=False),
    x="RepeatRatePct",
    y="CountryGroup",
    ax=axes[0, 0],
    palette="Blues_r",
)
axes[0, 0].set_title("Repeat Rate by Country Group")
axes[0, 0].set_xlabel("Repeat rate (%)")
axes[0, 0].set_ylabel("Country group")

sns.barplot(
    data=value_segment_table,
    x="FirstOrderValueSegment",
    y="RepeatRatePct",
    ax=axes[0, 1],
    palette="Greens",
)
axes[0, 1].set_title("Repeat Rate by First-Order Value Segment")
axes[0, 1].set_xlabel("First-order value segment")
axes[0, 1].set_ylabel("Repeat rate (%)")

sns.histplot(customer_summary.loc[customer_summary["RepeatCustomer"], "MedianGapDays"].dropna(), bins=30, ax=axes[1, 0], color="coral")
axes[1, 0].set_title("Median Interpurchase Gap Among Repeat Customers")
axes[1, 0].set_xlabel("Median gap (days)")
axes[1, 0].set_ylabel("Customers")

sns.heatmap(retention_rates.sort_index().head(12) * 100, annot=True, fmt=".0f", cmap="YlGnBu", ax=axes[1, 1])
axes[1, 1].set_title("Monthly Cohort Retention Rate (%)")
axes[1, 1].set_xlabel("Months since first order")
axes[1, 1].set_ylabel("Cohort month")

plt.tight_layout()
plt.show()

In [13]:
repeat_window = pd.DataFrame(
    {
        "Window": ["Within 30 days", "Within 60 days", "Within 90 days"],
        "PctCustomers": [
            customer_summary["RepeatWithin30Days"].mean() * 100,
            customer_summary["RepeatWithin60Days"].mean() * 100,
            customer_summary["RepeatWithin90Days"].mean() * 100,
        ],
    }
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=repeat_window, x="Window", y="PctCustomers", ax=axes[0], palette="rocket")
axes[0].set_title("Customers Returning Inside Key Time Windows")
axes[0].set_xlabel("")
axes[0].set_ylabel("Customers (%)")

sns.scatterplot(
    data=customer_summary,
    x="FirstOrderValue",
    y="OrderCount",
    hue="RepeatCustomer",
    alpha=0.6,
    ax=axes[1],
)
axes[1].set_title("First-Order Value vs Total Order Count")
axes[1].set_xlabel("First-order value")
axes[1].set_ylabel("Order count")

plt.tight_layout()
plt.show()

## Key Findings

The next cell translates the computed outputs into compact business-facing findings grounded in the actual dataset rather than prewritten claims.

In [14]:
best_country_row = country_repeat[country_repeat["Customers"] >= MIN_COUNTRY_CUSTOMERS].sort_values(
    "RepeatRatePct", ascending=False
).iloc[0]
best_value_row = value_segment_table.sort_values("RepeatRatePct", ascending=False).iloc[0]
best_breadth_row = breadth_segment_table.sort_values("RepeatRatePct", ascending=False).iloc[0]
strongest_profile = segment_matrix.stack().sort_values(ascending=False).index[0]

print("Key findings")
print("=" * 80)
print(
    f"1. {customer_summary['RepeatCustomer'].mean() * 100:.1f}% of customers place at least two orders, "
    f"and {customer_summary['ThreePlusOrders'].mean() * 100:.1f}% reach three or more orders."
)
print(
    f"2. The median interpurchase gap is {orders['GapDays'].dropna().median():.0f} days, "
    f"with 90% of gaps occurring inside {orders['GapDays'].dropna().quantile(0.90):.0f} days."
)
print(
    f"3. Among country groups with at least {MIN_COUNTRY_CUSTOMERS} customers, "
    f"{best_country_row['CountryGroup']} shows the strongest repeat rate at {best_country_row['RepeatRatePct']:.1f}%."
)
print(
    f"4. Customers in the {best_value_row['FirstOrderValueSegment']} first-order value segment repeat most often "
    f"({best_value_row['RepeatRatePct']:.1f}%), while the {best_breadth_row['FirstOrderBreadthSegment']} SKU-breadth segment "
    f"also leads on repeat rate ({best_breadth_row['RepeatRatePct']:.1f}%)."
)
print(
    f"5. The strongest combined profile is {strongest_profile[0]} breadth with {strongest_profile[1]} first-order value, "
    f"showing a repeat rate of {segment_matrix.loc[strongest_profile[0], strongest_profile[1]]:.1f}%."
)

Key findings
1. 65.6% of customers place at least two orders, and 46.3% reach three or more orders.
2. The median interpurchase gap is 21 days, with 90% of gaps occurring inside 103 days.
3. Among country groups with at least 30 customers, Germany shows the strongest repeat rate at 72.3%.
4. Customers in the High first-order value segment repeat most often (70.3%), while the Wide SKU-breadth segment also leads on repeat rate (68.3%).
5. The strongest combined profile is Wide breadth with High first-order value, showing a repeat rate of 71.4%.


## Limitations

1. The dataset does not include marketing channel, campaign exposure, or customer demographics, so repurchase drivers are inferred from transaction behavior and geography only.
2. Geography is heavily dominated by the United Kingdom, which can bias overall repeat-rate averages.
3. The workbook records completed invoice events, not browsing or cart behavior, so pre-purchase intent is invisible.
4. Returned or cancelled orders are removed to keep repeat behavior aligned with successful commercial transactions.
5. Observed retention patterns are descriptive and should not be treated as causal without controlled experimentation.

## Recommendations / Next Steps

1. Trigger a structured post-purchase retention program inside the first 30 days, because that window captures a large share of second orders.
2. Prioritize higher-value and broader first-order baskets for cross-sell and replenishment campaigns; they show stronger repeat propensity.
3. Build country-specific follow-up playbooks for the largest non-UK markets rather than assuming one retention cadence fits all.
4. Add acquisition channel and campaign data to separate merchandising effects from marketing effects.
5. Move from descriptive segmentation to predictive retention scoring using first-order features as model inputs.

## Common Mistakes

1. Treating raw line items as orders instead of aggregating invoices first.
2. Including cancellation invoices in repeat-rate calculations.
3. Measuring interpurchase gaps without removing customers who only bought once.
4. Comparing tiny country groups to large markets without a minimum sample threshold.
5. Assuming a high repeat rate automatically implies healthy timing; slow repeat cycles can still weaken cash flow.

## Mini Challenge

1. Recompute repeat rates using only UK customers and compare the result with the global customer base.
2. Test whether discount-heavy invoices have shorter or longer second-purchase gaps once discount features are available.
3. Replace simple country groups with an RFM-style segmentation and compare repeat windows.
4. Estimate the probability of second purchase inside 60 days using logistic regression on first-order features.
5. Extend the cohort table into revenue retention, not just customer retention.

## Final Summary

Repeat behavior in this dataset is substantial, but not evenly distributed. Customers with larger and broader first orders are more likely to return, repeat purchase timing is concentrated inside a relatively short window, and retention decays by cohort after acquisition in a measurable pattern. Those findings point to a practical retention strategy: use first-order basket signals and early post-purchase timing to focus limited CRM effort where repurchase odds are highest.

In [15]:
final_summary = pd.DataFrame(
    {
        "Metric": [
            "Repeat customers",
            "Customers with 3+ orders",
            "Median interpurchase gap",
            "Best country-group repeat rate",
            "Best first-order value segment repeat rate",
        ],
        "Value": [
            f"{customer_summary['RepeatCustomer'].mean() * 100:.1f}%",
            f"{customer_summary['ThreePlusOrders'].mean() * 100:.1f}%",
            f"{orders['GapDays'].dropna().median():.0f} days",
            f"{country_repeat[country_repeat['Customers'] >= MIN_COUNTRY_CUSTOMERS].sort_values('RepeatRatePct', ascending=False).iloc[0]['RepeatRatePct']:.1f}%",
            f"{value_segment_table.sort_values('RepeatRatePct', ascending=False).iloc[0]['RepeatRatePct']:.1f}%",
        ],
    }
)
display(final_summary)

,Metric,Value
0,Repeat customers,65.6%
1,Customers with 3+ orders,46.3%
2,Median interpurchase gap,21 days
3,Best country-group repeat rate,72.3%
4,Best first-order value segment repeat rate,70.3%
